# Embeddings des avis antiparasitaires US, e5-base sur GPU

Projet ReviewFlowz, 2026-09-30. Même modèle et même paramétrage que
`etudes-ponctuelles/2026-09-21-texte-embedding-03B/01_embedding.py`, pour que les vecteurs se
comparent aux études précédentes et que le seuil de copie (0,97) reste valable.

**Avant de commencer**
1. Exécution → Modifier le type d'exécution → **GPU T4**.
2. Déposer `pest_textes_colab.parquet` (dans `data/local/` du projet, 16 Mo) dans le dossier
   `reviewflowz_pest` du Drive **du compte Cartelis**.
3. Lancer les cellules dans l'ordre. En cas de coupure, relancer tout : les tranches déjà
   calculées sont relues depuis le Drive.

**Après** : télécharger `pest_e5base.parquet`, le placer dans `data/embeddings/` du projet,
lancer `verifier_colab.py`, puis effacer les deux fichiers du Drive (les textes sont des données du client).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "pas de GPU : changer le type d'exécution"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/reviewflowz_pest"
import os
ENTREE = f"{DOSSIER}/pest_textes_colab.parquet"
TRANCHES = f"{DOSSIER}/tranches"
os.makedirs(TRANCHES, exist_ok=True)
assert os.path.exists(ENTREE), f"{ENTREE} absent"

In [ ]:
# Paramétrage identique au 21/09 (01_embedding.py)
MODELE = "intfloat/multilingual-e5-base"
PREFIXE = "query: "          # demandé par e5
LONGUEUR_MAX = 512           # longueur du modèle, aucun texte tronqué en plus
NORMALISER = True            # vecteurs de norme 1 : le produit est la similarité
# Propre au GPU
TAILLE_LOT = 256
TAILLE_TRANCHE = 20_000      # une sauvegarde sur le Drive toutes les 20 000 textes
DEMI_PRECISION = False       # False : calcul en float32, comme en local

In [ ]:
import numpy as np, pandas as pd
df = pd.read_parquet(ENTREE)
print(len(df), "avis,", df["review_id"].nunique(), "identifiants")
# Chaque texte différent n'est encodé qu'une fois, puis recopié sur ses doublons.
uniques = pd.Series(df["texte"].unique())
ordre = np.argsort(uniques.str.len().to_numpy(), kind="stable")   # lots de longueur voisine
textes = [PREFIXE + t for t in uniques.iloc[ordre]]
print(len(textes), "textes différents à encoder")

In [ ]:
from sentence_transformers import SentenceTransformer
import sentence_transformers
modele = SentenceTransformer(MODELE, device="cuda")
modele.max_seq_length = LONGUEUR_MAX
if DEMI_PRECISION:
    modele.half()
print(sentence_transformers.__version__, modele.max_seq_length)

In [ ]:
import os, time
n = len(textes)
depart = time.time()
# Les tranches déjà présentes sont reprises : les effacer si le paramétrage a changé.
for i, debut in enumerate(range(0, n, TAILLE_TRANCHE)):
    chemin = f"{TRANCHES}/tranche_{i:03d}.npy"
    if os.path.exists(chemin):
        continue
    v = modele.encode(textes[debut:debut + TAILLE_TRANCHE], batch_size=TAILLE_LOT,
                      normalize_embeddings=NORMALISER, convert_to_numpy=True,
                      show_progress_bar=False)
    np.save(chemin, v.astype("float32"))
    fait = min(debut + TAILLE_TRANCHE, n)
    print(f"{fait:>7}/{n}  {(time.time() - depart) / 60:5.1f} min", flush=True)
print("encodage terminé")

In [ ]:
import os, time, torch, sentence_transformers
import pyarrow as pa, pyarrow.parquet as pq
fichiers = sorted(f for f in os.listdir(TRANCHES) if f.endswith(".npy"))
v_tries = np.vstack([np.load(f"{TRANCHES}/{f}") for f in fichiers])
assert len(v_tries) == len(textes), (len(v_tries), len(textes))
v_uniques = np.empty_like(v_tries)
v_uniques[ordre] = v_tries                                  # remise dans l'ordre de `uniques`
indice = pd.Index(uniques).get_indexer(df["texte"])        # texte -> ligne de v_uniques
assert (indice >= 0).all()
vecteurs = v_uniques[indice].astype("float16")              # float16, comme 03B_e5base.parquet
normes = np.linalg.norm(vecteurs.astype("float32"), axis=1)
print("normes : min", normes.min(), "max", normes.max())

d = vecteurs.shape[1]
valeurs = pa.array(vecteurs.reshape(-1), type=pa.float16())
decalages = pa.array(np.arange(0, len(df) * d + 1, d, dtype=np.int32))
table = pa.table({"review_id": pa.array(df["review_id"]),
                  "vecteur": pa.ListArray.from_arrays(decalages, valeurs)})
table = table.replace_schema_metadata({
    "modele": MODELE, "prefixe": PREFIXE, "longueur_max": str(LONGUEUR_MAX),
    "demi_precision": str(DEMI_PRECISION), "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
    "date": time.strftime("%Y-%m-%d")})
SORTIE = f"{DOSSIER}/pest_e5base.parquet"
pq.write_table(table, SORTIE, compression="zstd")
print(SORTIE, round(os.path.getsize(SORTIE) / 1024**2), "Mo,", table.num_rows, "lignes,", d, "dimensions")

In [ ]:
# Téléchargement direct (sinon le récupérer depuis le Drive)
from google.colab import files
files.download(SORTIE)

**Nettoyage** une fois `pest_e5base.parquet` vérifié en local : supprimer le dossier
`reviewflowz_pest` du Drive (textes, tranches et vecteurs), puis vider la corbeille du Drive.